## Washington capital bike rental

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load data

In [ ]:

s3_data_path = "s3://mlops-project-bucket-602343785232-ap-southeast-1-an/extract_data/bike_rental_y_2022_2025_m_1_12.csv"

df = pd.read_csv(s3_data_path)



### General info

In [ ]:

df.head(5)

In [ ]:

df.info()

In [ ]:
df.describe()

### Duplicate hourly time point

In [ ]:

time_keys = ["Year", "Month", "Day", "Hour"]
duplicate_time_mask = df.duplicated(subset=time_keys, keep=False)
duplicate_time_points_df = df.loc[duplicate_time_mask].copy()
print(f"Duplicate hourly rows found: {len(duplicate_time_points_df):,}")
df = df.drop_duplicates(subset=time_keys, keep="first").reset_index(drop=True)
print(f"Rows after removing duplicate time points: {len(df):,}")


### Handle missing value

In [ ]:

missing_temperature_mask = df["temperature"].isna()
missing_temperature_df = df.loc[missing_temperature_mask].copy()
print(f"Rows missing temperature: {len(missing_temperature_df):,}")
display(missing_temperature_df.head())


### Numerical Analysis

In [ ]:
df['total_classic_bike_count'] = df['classic_bike_casual_count'] + df['classic_bike_member_count']
df['total_docked_bike_count'] = df['docked_bike_casual_count'] + df['docked_bike_member_count'] 
df['total_electric_bike_count'] =  df['electric_bike_casual_count'] + df['electric_bike_member_count']

df['total_bike_rental_count'] = df['total_classic_bike_count'] + df['total_docked_bike_count'] + df['total_electric_bike_count']

In [ ]:
df[['Year','Month','Day','Hour']].describe()

In [ ]:
df.columns

In [ ]:
df[['Year','Month','Day','Hour']].describe()

In [ ]:
df[['classic_bike_casual_count','docked_bike_casual_count','electric_bike_casual_count']].describe()

In [ ]:
df[['classic_bike_member_count','docked_bike_member_count','electric_bike_member_count']].describe()

In [ ]:
df[['total_classic_bike_count','total_docked_bike_count','total_electric_bike_count','total_bike_rental_count']].describe()

In [ ]:

classic_correlation = df[[
    "classic_bike_casual_count",
    "classic_bike_member_count",
    "total_bike_rental_count",
]].corr()
display(classic_correlation)

electric_correlation = df[[
    "electric_bike_casual_count",
    "electric_bike_member_count",
    "total_bike_rental_count",
]].corr()
display(electric_correlation)

docked_correlation = df[[
    "docked_bike_casual_count",
    "docked_bike_member_count",
    "total_bike_rental_count",
]].corr()
display(docked_correlation)


In [ ]:

total_bike_correlation = df[[
    "total_classic_bike_count",
    "total_docked_bike_count",
    "total_electric_bike_count",
    "total_bike_rental_count",
]].corr()[["total_bike_rental_count"]].sort_values(
    "total_bike_rental_count", ascending=False
)
display(total_bike_correlation)


In [ ]:
# AGENT TASK
# Analyze temperature correlation with each bike-rental count column.
bike_columns = [
    column for column in df.columns
    if "bike" in column.lower() or column.endswith("_count")
]
temperature_bike_correlation = (
    df[bike_columns + ["temperature"]]
    .corr()["temperature"]
    .drop("temperature")
    .sort_values()
    .rename("temperature_correlation")
)
display(temperature_bike_correlation.to_frame())




### Time analysis (Monthly) hue = year

In [ ]:

bike_columns = [
    column for column in df.columns
    if "bike" in column.lower() or column.endswith("_count")
]
time_bike_correlation = (
    df[["Month", "Hour", *bike_columns]]
    .corr()
    .loc[["Month", "Hour"], bike_columns]
)
display(time_bike_correlation)

plt.figure(figsize=(12, 4))  # Widen the plot so feature labels remain readable.
sns.heatmap(
    time_bike_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"},
)
plt.title("Correlation of Month and Hour with Bike Rentals")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

monthly_year_average = (
    df.groupby(["Month", "Year"], as_index=False)["total_bike_rental_count"]
    .sum()
    .rename(columns={"total_bike_rental_count": "Total Rental"})
)
plt.figure(figsize=(8, 4))  # Keep the monthly comparison compact.
sns.barplot(
    data=monthly_year_average,
    x="Month",
    y="Total Rental",
    hue="Year",
    palette="tab10",
)
plt.title("Monthly Total Bike Rentals by Year")
plt.xlabel("Month")
plt.ylabel("Bike Rentals")
plt.legend(title="Year")
plt.tight_layout()
plt.show()


In [ ]:

bike_type_totals = {
    "Classic bike": "total_classic_bike_count",
    "Docked bike": "total_docked_bike_count",
    "Electric bike": "total_electric_bike_count",
}
monthly_bike_totals = (
    df.groupby(["Year", "Month"], as_index=False)[list(bike_type_totals.values())]
    .sum()
    .melt(
        id_vars=["Year", "Month"],
        var_name="Bike Type",
        value_name="Total Rentals",
    )
    .replace({"Bike Type": {value: key for key, value in bike_type_totals.items()}})
)

monthly_plot = sns.catplot(
    data=monthly_bike_totals,
    x="Month",
    y="Total Rentals",
    hue="Year",
    col="Bike Type",
    col_wrap=3,
    kind="bar",
    palette="tab10",
    errorbar=None,
    height=3.2,
    aspect=1.2,
)
monthly_plot.set_axis_labels("Month", "Total Rentals")
monthly_plot.set_titles("{col_name}")
monthly_plot.figure.suptitle("Monthly Total Bike Rentals by Year and Bike Type", y=1.03)
monthly_plot._legend.set_bbox_to_anchor((1.02, 0.5))  # Move the legend outside the panels.
monthly_plot.figure.tight_layout(rect=[0, 0, 0.86, 1])  # Reserve space for the legend.
plt.show()


### Time analysis (weekly windows by year) (Hue = year)


In [ ]:

rental_columns = [
    "total_bike_rental_count",
    "total_classic_bike_count",
    "total_docked_bike_count",
    "total_electric_bike_count",
]

daily_rentals = (
    df.groupby(["Year", "Month", "Day"], as_index=False)[rental_columns]
    .sum()
    .assign(
        Date=lambda data: pd.to_datetime(
            data[["Year", "Month", "Day"]].rename(
                columns={"Year": "year", "Month": "month", "Day": "day"}
            )
        )
    )
)

# Number consecutive seven-day windows from January 1 separately for each year.
year_starts = pd.to_datetime(daily_rentals["Year"].astype(str) + "-01-01")
daily_rentals["week_number"] = (
    daily_rentals["Date"].sub(year_starts).dt.days // 7 + 1
)


In [ ]:

weekly_rentals = (
    daily_rentals.groupby(["Year", "week_number"], as_index=False)["total_bike_rental_count"]
    .sum()
    .rename(columns={"total_bike_rental_count": "total_rentals"})
)
# display(weekly_rentals)

plt.figure(figsize=(14, 3.5))  # Reduce height while showing all years.
sns.barplot(
    data=weekly_rentals,
    x="week_number",
    y="total_rentals",
    hue="Year",
    palette="tab10",
    errorbar=None,
)
plt.title("Weekly Total Bike Rentals by Year")
plt.xlabel("Seven-Day Window Number")
plt.ylabel("Total Rentals")
plt.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:

weekly_bike_rentals = (
    daily_rentals.groupby(["Year", "week_number"], as_index=False)[list(bike_type_totals.values())]
    .sum()
    .melt(
        id_vars=["Year", "week_number"],
        var_name="Bike Type",
        value_name="Total Rentals",
    )
    .replace({"Bike Type": {value: key for key, value in bike_type_totals.items()}})
)
# display(weekly_bike_rentals)

# Use one compact panel per bike type so all years remain comparable in limited space.
weekly_plot = sns.catplot(
    data=weekly_bike_rentals,
    x="week_number",
    y="Total Rentals",
    hue="Year",
    col="Bike Type",
    col_wrap=1,  # Stack the bike-type panels vertically.
    col_order=list(bike_type_totals),
    kind="bar",
    palette="tab10",
    errorbar=None,
    height=2.2,  # Keep each vertically stacked panel compact.
    aspect=5.0,  # Expand each stacked panel so the weekly bars are readable.
)
weekly_plot.set_axis_labels("Seven-Day Window Number", "Total Rentals")
weekly_plot.set_titles("{col_name}")
weekly_plot.figure.suptitle("Weekly Bike Rentals by Year and Bike Type", y=1.03)
weekly_plot._legend.set_bbox_to_anchor((1.02, 0.5))  # Move the year legend outside the panels.
for axis in weekly_plot.axes.flat:
    axis.tick_params(axis="x", labelrotation=90, labelsize=6)
weekly_plot.figure.tight_layout(rect=[0, 0, 0.88, 1])  # Reserve space for the legend.
plt.show()


### Hourly Analysis

#### Question 1 - Are there common pattern for bike rantal hourly across all year
##### Q1 ans : Pattern are the same across yearly during the same month. There are peak and low hour. However, The magnitudes are not the same

#### Question 2 - Does temperature affect hourly bike rental 
##### Q2 ans : nope

### Noteable finding

##### Hourly classic Bike demand is stable across different year meanwhile hourly electric bike are higher in 2024 and 2025 while remain lower on 2022 and 2023



In [ ]:
# AGENT TASK
# Plot raw hourly rental observations for each month, with Year as the comparison hue.
hourly_rentals = df[
    ["Year", "Month", "Hour", "total_bike_rental_count"]
].rename(columns={"total_bike_rental_count": "Bike Rentals"})
# Keep every day-hour row in the input; seaborn aggregates repeated hours by mean.
# display(hourly_rentals)

# Arrange the 12 months in three rows and four columns for direct month-to-month comparison.
hourly_plot = sns.relplot(
    data=hourly_rentals,
    x="Hour",
    y="Bike Rentals",
    hue="Year",
    col="Month",
    col_order=list(range(1, 13)),
    col_wrap=4,
    kind="line",
    palette="tab10",
    height=2.3,
    aspect=1.45,
)
hourly_plot.set_axis_labels("Hour of Day", "Average Bike Rentals")
hourly_plot.set_titles("Month {col_name}")
hourly_plot.figure.suptitle("Average Hourly Bike Rentals by Month and Year", y=1.02)
hourly_plot._legend.set_bbox_to_anchor((1.02, 0.5))  # Move the year legend outside the month panels.
for axis in hourly_plot.axes.flat:
    axis.set_xticks([0, 4, 8, 12, 16, 20, 23])  # Show regular intervals and the final hour 23.
hourly_plot.figure.tight_layout(rect=[0, 0, 0.88, 1])  # Reserve space for the legend.
plt.show()


In [ ]:
# AGENT TASK
# Plot hourly classic-bike rentals, electric-bike rentals, and temperature for each month, with Year as hue.
monthly_hourly_data = df[
    [
        "Year",
        "Month",
        "Hour",
        "total_classic_bike_count",
        "total_electric_bike_count",
        "temperature",
    ]
].rename(
    columns={
        "total_classic_bike_count": "Classic Bike Rentals",
        "total_electric_bike_count": "Electric Bike Rentals",
    }
)
year_order = sorted(monthly_hourly_data["Year"].dropna().unique())

# Create one row per month: classic bike, electric bike, then temperature.
fig, axes = plt.subplots(
    nrows=12,
    ncols=3,
    figsize=(18, 30),
    sharex=True,
    sharey=False,  # Let each panel use its own y-axis range.
)
year_handles = year_labels = None
for row, month in enumerate(range(1, 13)):
    month_data = monthly_hourly_data[monthly_hourly_data["Month"] == month]
    classic_axis, electric_axis, temperature_axis = axes[row]

    # Seaborn receives raw rows and averages repeated hours by Year within each month.
    sns.lineplot(
        data=month_data,
        x="Hour",
        y="Classic Bike Rentals",
        hue="Year",
        hue_order=year_order,
        palette="tab10",
        legend=(row == 0),
        ax=classic_axis,
    )
    sns.lineplot(
        data=month_data,
        x="Hour",
        y="Electric Bike Rentals",
        hue="Year",
        hue_order=year_order,
        palette="tab10",
        legend=False,
        ax=electric_axis,
    )
    sns.lineplot(
        data=month_data,
        x="Hour",
        y="temperature",
        hue="Year",
        hue_order=year_order,
        palette="tab10",
        legend=False,
        ax=temperature_axis,
    )

    classic_axis.set_title(f"Month {month} - Classic")
    electric_axis.set_title(f"Month {month} - Electric")
    temperature_axis.set_title(f"Month {month} - Temperature")
    classic_axis.set_ylabel("Average Rentals")
    electric_axis.set_ylabel("Average Rentals")
    temperature_axis.set_ylabel("Temperature (°C)")
    for axis in (classic_axis, electric_axis, temperature_axis):
        axis.set_xlabel("Hour of Day")
        axis.set_xticks([0, 4, 8, 12, 16, 20, 23])  # Show regular intervals and the final hour 23.

    if row == 0 and classic_axis.legend_ is not None:
        year_handles, year_labels = classic_axis.get_legend_handles_labels()
        classic_axis.legend_.remove()

fig.suptitle(
    "Hourly Classic Bike, Electric Bike, and Temperature by Month and Year (Y-axis scales vary by panel)",
    y=0.995,
)
if year_handles:
    fig.legend(year_handles, year_labels, title="Year", loc="upper right", bbox_to_anchor=(0.995, 0.995))
fig.tight_layout(rect=[0, 0, 0.94, 0.99])
plt.show()


### Year 2025 vs 2022 analysis

##### Assumption lower hourly average temperature during 2025 lead to lower classic bike usage

In [ ]:
# AGENT TASK
# Compare each year with the 2022 baseline using hourly classic-bike and temperature differences.
comparison_keys = ["Month", "Day", "Hour"]
comparison_columns = [*comparison_keys, "total_classic_bike_count", "temperature"]
comparison_years = [2023, 2024, 2025]
daytime_rows = df["Hour"].between(6, 20)  # Include only hours 6 through 20.

baseline_2022 = (
    df.loc[(df["Year"] == 2022) & daytime_rows, comparison_columns]
    .rename(
        columns={
            "total_classic_bike_count": "classic_bike_rentals_2022",
            "temperature": "temperature_2022",
        }
    )
)

difference_frames = []
for comparison_year in comparison_years:
    year_data = (
        df.loc[(df["Year"] == comparison_year) & daytime_rows, comparison_columns]
        .rename(
            columns={
                "total_classic_bike_count": "classic_bike_rentals_comparison",
                "temperature": "temperature_comparison",
            }
        )
    )

    # Match the same calendar day and hour against 2022 without changing the original dataframe.
    year_differences = year_data.merge(
        baseline_2022,
        on=comparison_keys,
        how="inner",
    )
    year_differences["classic_bike_rental_difference"] = (
        year_differences["classic_bike_rentals_comparison"]
        - year_differences["classic_bike_rentals_2022"]
    )
    year_differences["temperature_difference"] = (
        year_differences["temperature_comparison"]
        - year_differences["temperature_2022"]
    )
    year_differences["Comparison Year"] = comparison_year
    difference_frames.append(year_differences)

difference_data = pd.concat(difference_frames, ignore_index=True)


# Calculate one temperature/rental difference correlation per comparison year and month.
monthly_correlation_rows = []
for (comparison_year, month), month_data in difference_data.groupby(["Comparison Year", "Month"]):
    monthly_correlation_rows.append(
        {
            "Comparison Year": comparison_year,
            "Month": month,
            "Correlation": month_data["classic_bike_rental_difference"].corr(
                month_data["temperature_difference"]
            ),
        }
    )
monthly_difference_correlation = pd.DataFrame(monthly_correlation_rows).sort_values(
    ["Month", "Comparison Year"]
)

# Compare the monthly relationships for 2023, 2024, and 2025 against 2022.
plt.figure(figsize=(11, 4))
sns.barplot(
    data=monthly_difference_correlation,
    x="Month",
    y="Correlation",
    hue="Comparison Year",
    palette="tab10",
    errorbar=None,
)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylim(-1, 1)
plt.title("Monthly Correlation of Temperature and Classic Rental Differences vs 2022")
plt.xlabel("Month")
plt.ylabel("Pearson Correlation")
plt.legend(title="Comparison Year", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
